# 🦋 CBOA-LSTM Optimization for Network Intrusion Detection
## تحسين شبكة LSTM باستخدام خوارزمية الفراشة الفوضوية (CBOA)

---

### 📌 نظرة عامة على الكود:
هذا الكود يستخدم **Chaotic Butterfly Optimization Algorithm (CBOA)** لتحسين أوزان شبكة **LSTM** لاكتشاف الاختراقات في الشبكات (Network Intrusion Detection) باستخدام **KDD Dataset**.

### المراحل:
1. **تحميل ومعالجة البيانات** (KDD Dataset)
2. **بناء شبكة LSTM**
3. **تطبيق CBOA لتحسين الأوزان**
4. **تقييم النتائج** (Accuracy, Confusion Matrix, etc.)

---
### ⚠️ ملاحظات مهمة وأخطاء في الكود الأصلي:
- في `CBOAoptimization.m`: الـ `ChaosVec` يُنشأ كـ `(1, MaxIt)` لكن يتم الوصول إليه كـ `ChaosVec(it)` بدون indexing صحيح للصفوف العشرة.
- في `Nominal2Numbersbinary.m`: وجود `pdf_p`, `pdf_f`, `pdf_s` يتم تعريفها ولكن لا تُستخدم (dead code).
- في `CBOAoptimization.m`: `particle(i).Position = nVar` يجعل كل الـ particles تبدأ من نفس المكان بدلاً من تهيئة عشوائية.
- `Nominal2Numbersbinary.m`: دالة اسمها `Nominal2Numbersbinary` لكن تعريفها الداخلي `Nominal2Numbers` (اسم مختلف - خطأ!).
- في `confusion1.m`: الكود يشترط أن class lists في actual و predict متطابقة، مما يسبب error إذا prediction لم تشمل كل الكلاسات.
---

## 📦 Cell 1: تثبيت المكتبات المطلوبة

In [ ]:
# تثبيت المكتبات المطلوبة
!pip install torch scikit-learn pandas numpy matplotlib seaborn tqdm

## 📚 Cell 2: Import المكتبات

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import copy
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Sklearn
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, matthews_corrcoef, f1_score
)

from tqdm import tqdm

print("✅ جميع المكتبات تم تحميلها بنجاح!")
print(f"PyTorch version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

---
## 🌀 Cell 3: دالة الـ Chaotic Maps
### (ترجمة `chaos.m`)

**الشرح:** هذه الدالة تولّد أرقاماً فوضوية (chaotic numbers) بدلاً من الأرقام العشوائية العادية.
الفوضى تساعد الخوارزمية على استكشاف أفضل للفضاء وتجنب الوقوع في minimum محلي.

**الأنواع المتاحة (Index):**
- 1: Chebyshev | 2: Circle | 3: Gauss | 4: Iterative | 5: Logistic
- 6: Piecewise | 7: Sine | 8: Singer | 9: Sinusoidal | 10: Tent

In [ ]:
def chaos(index, value, max_iter):
    """
    توليد متتالية فوضوية.
    
    Parameters:
        index (int): نوع الخريطة الفوضوية (1-11)
        value (float): قيمة المقياس
        max_iter (int): عدد التكرارات
    
    Returns:
        np.array: متتالية فوضوية بطول max_iter
    """
    x = np.zeros(max_iter + 1)
    G = np.zeros(max_iter)
    x[0] = 0.7  # القيمة الابتدائية

    if index == 1:  # Chebyshev map
        for i in range(max_iter):
            x[i+1] = np.cos((i+1) * np.arccos(x[i]))
            G[i] = ((x[i] + 1) * value) / 2

    elif index == 2:  # Circle map
        a, b = 0.5, 0.2
        for i in range(max_iter):
            x[i+1] = (x[i] + b - (a / (2 * np.pi)) * np.sin(2 * np.pi * x[i])) % 1
            G[i] = x[i] * value

    elif index == 3:  # Gauss/mouse map
        for i in range(max_iter):
            x[i+1] = 0.0 if x[i] == 0 else (1 / x[i]) % 1
            G[i] = x[i] * value

    elif index == 4:  # Iterative map
        a = 0.7
        for i in range(max_iter):
            x[i+1] = np.sin((a * np.pi) / x[i]) if x[i] != 0 else 0
            G[i] = ((x[i] + 1) * value) / 2

    elif index == 5:  # Logistic map
        a = 4
        for i in range(max_iter):
            x[i+1] = a * x[i] * (1 - x[i])
            G[i] = x[i] * value

    elif index == 6:  # Piecewise map
        P = 0.4
        for i in range(max_iter):
            if 0 <= x[i] < P:
                x[i+1] = x[i] / P
            elif P <= x[i] < 0.5:
                x[i+1] = (x[i] - P) / (0.5 - P)
            elif 0.5 <= x[i] < 1 - P:
                x[i+1] = (1 - P - x[i]) / (0.5 - P)
            elif 1 - P <= x[i] < 1:
                x[i+1] = (1 - x[i]) / P
            G[i] = x[i] * value

    elif index == 7:  # Sine map
        for i in range(max_iter):
            x[i+1] = np.sin(np.pi * x[i])
            G[i] = x[i] * value

    elif index == 8:  # Singer map
        u = 1.07
        for i in range(max_iter):
            x[i+1] = u * (7.86*x[i] - 23.31*x[i]**2 + 28.75*x[i]**3 - 13.302875*x[i]**4)
            G[i] = x[i] * value

    elif index == 9:  # Sinusoidal map
        for i in range(max_iter):
            x[i+1] = 2.3 * x[i]**2 * np.sin(np.pi * x[i])
            G[i] = x[i] * value

    elif index == 10:  # Tent map
        x[0] = 0.6
        for i in range(max_iter):
            x[i+1] = x[i] / 0.7 if x[i] < 0.7 else (10/3) * (1 - x[i])
            G[i] = x[i] * value

    elif index == 11:  # Normal random (baseline)
        G = np.random.rand(max_iter)

    return G


# اختبار ورسم الخرائط الفوضوية
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
map_names = ['Chebyshev', 'Circle', 'Gauss', 'Iterative', 'Logistic',
             'Piecewise', 'Sine', 'Singer', 'Sinusoidal', 'Tent']

for idx, (ax, name) in enumerate(zip(axes.flat, map_names)):
    try:
        seq = chaos(idx+1, 1, 100)
        ax.plot(seq, color=plt.cm.tab10(idx/10), linewidth=1)
        ax.set_title(name, fontsize=9)
        ax.set_xlabel('Iteration')
        ax.grid(True, alpha=0.3)
    except Exception as e:
        ax.set_title(f"{name}\n(Error)")

plt.suptitle('🌀 الخرائط الفوضوية العشر - Chaotic Maps', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("✅ دالة chaos تعمل بشكل صحيح")

---
## 🔢 Cell 4: تحويل البيانات الاسمية إلى أرقام
### (ترجمة `Nominal2Numbersbinary.m` و `Nominal2Numbersmulti22.m`)

**الشرح:** بيانات KDD تحتوي على أعمدة نصية (protocol_type, service, flag) يجب تحويلها لأرقام.

- **Binary Mode**: normal=1, attack=2
- **Multi-class Mode**: 5 تصنيفات (normal, DOS, Probe, R2L, U2R) مع One-Hot Encoding

In [ ]:
# ===============================================================
# تعريفات الخريطة - نفس القيم في الكود الأصلي
# ===============================================================
PROTOCOL_TYPES = ['tcp', 'udp', 'icmp']

FLAGS = ['SF', 'S0', 'REJ', 'RSTR', 'SH', 'RSTO', 'S1', 'RSTOS0', 'S3', 'S2', 'OTH']

SERVICES = [
    'ftp_data', 'other', 'private', 'http', 'remote_job',
    'name', 'netbios_ns', 'eco_i', 'mtp', 'telnet', 'finger',
    'domain_u', 'supdup', 'uucp_path', 'Z39_50', 'smtp', 'csnet_ns',
    'uucp', 'netbios_dgm', 'urp_i', 'auth', 'domain', 'ftp', 'bgp',
    'ldap', 'ecr_i', 'gopher', 'vmnet', 'systat', 'http_443', 'efs',
    'whois', 'imap4', 'iso_tsap', 'echo', 'klogin', 'link', 'sunrpc',
    'login', 'kshell', 'sql_net', 'time', 'hostnames', 'exec', 'ntp_u',
    'discard', 'nntp', 'courier', 'ctf', 'ssh', 'daytime', 'shell',
    'netstat', 'pop_3', 'nnsp', 'IRC', 'pop_2', 'printer', 'tim_i',
    'pm_dump', 'red_i', 'netbios_ssn', 'rje', 'X11', 'urh_i',
    'http_8001', 'aol', 'http_2784', 'tftp_u', 'harvest'
]

# خريطة هجمات DOS
DOS_ATTACKS = ['smurf', 'neptune', 'mailbomb', 'back', 'teardrop', 'pod',
               'apache2', 'udpstorm', 'processtable', 'land', 'worm']
# هجمات Probe
PROBE_ATTACKS = ['satan', 'ipsweep', 'portsweep', 'nmap', 'mscan', 'saint']
# هجمات R2L
R2L_ATTACKS = ['warezclient', 'guess_passwd', 'warezmaster', 'imap', 'ftp_write',
               'multihop', 'phf', 'spy', 'xlock', 'xsnoop', 'snmpguess',
               'snmpgetattack', 'httptunnel', 'sendmail', 'named']
# هجمات U2R
U2R_ATTACKS = ['buffer_overflow', 'rootkit', 'loadmodule', 'sqlattack', 'xterm', 'ps', 'perl']


def map_label_multiclass(label):
    """تصنيف الهجمات إلى 5 فئات."""
    label = str(label).strip().lower().rstrip('.')
    if label == 'normal':
        return 'normal'
    elif label in [a.lower() for a in DOS_ATTACKS]:
        return 'DOS'
    elif label in [a.lower() for a in PROBE_ATTACKS]:
        return 'Probe'
    elif label in [a.lower() for a in R2L_ATTACKS]:
        return 'R2L'
    elif label in [a.lower() for a in U2R_ATTACKS]:
        return 'U2R'
    else:
        return 'unknown'


def preprocess_kdd(filepath, mode='multi'):
    """
    تحميل ومعالجة بيانات KDD.
    
    Parameters:
        filepath (str): مسار الملف
        mode (str): 'binary' أو 'multi'
    
    Returns:
        X (np.array): المميزات بعد المعالجة
        y (np.array): التصنيفات
    """
    # أسماء الأعمدة من KDD Dataset
    col_names = [
        'duration','protocol_type','service','flag','src_bytes','dst_bytes',
        'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
        'num_compromised','root_shell','su_attempted','num_root','num_file_creations',
        'num_shells','num_access_files','num_outbound_cmds','is_host_login',
        'is_guest_login','count','srv_count','serror_rate','srv_serror_rate',
        'rerror_rate','srv_rerror_rate','same_srv_rate','diff_srv_rate',
        'srv_diff_host_rate','dst_host_count','dst_host_srv_count',
        'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
        'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
        'dst_host_rerror_rate','dst_host_srv_rerror_rate','label','difficulty'
    ]

    df = pd.read_csv(filepath, header=None, names=col_names)

    # إزالة عمود الـ difficulty
    df = df.drop(columns=['difficulty'], errors='ignore')

    # تحويل التصنيفات
    if mode == 'binary':
        df['label'] = df['label'].apply(
            lambda x: 'normal' if str(x).strip().lower().rstrip('.') == 'normal' else 'attack'
        )
    else:
        df['label'] = df['label'].apply(map_label_multiclass)

    # One-Hot Encoding للأعمدة الاسمية (بديل Pandas أفضل من MATLAB)
    df = pd.get_dummies(df, columns=['protocol_type', 'service', 'flag'])

    # فصل X و y
    y = df['label'].values
    X = df.drop(columns=['label']).values.astype(np.float32)

    return X, y


print("✅ دوال معالجة البيانات جاهزة")
print(f"عدد أنواع البروتوكولات: {len(PROTOCOL_TYPES)}")
print(f"عدد أنواع الـ Flag: {len(FLAGS)}")
print(f"عدد أنواع الـ Service: {len(SERVICES)}")

---
## 📊 Cell 5: Normalization (Min-Max)
### (ترجمة `Normalization.m`)

**الشرح:** تحويل القيم إلى نطاق [0, 1] لتسهيل تدريب الشبكة. الكود الأصلي يتجاهل التطبيع للأعمدة اللي قيمتها بين 0 و1 مسبقاً.

In [ ]:
def normalization(X):
    """
    Min-Max Normalization.
    الأعمدة اللي max <= 1 تتجاهل (زي الكود الأصلي).
    الأعمدة التانية بتتحول لنطاق [0, 1].
    
    ⚠️ ملاحظة: الكود الأصلي فيه bug - بيتحقق من max > 1 بس مش بيتعامل
    مع حالة maximum == minimum (قسمة على صفر).
    إصلاح: أضفنا تحقق من أن (max - min) > 0
    """
    X_norm = X.copy().astype(np.float32)
    
    for i in range(X.shape[1]):
        col = X[:, i]
        col_max = col.max()
        col_min = col.min()
        
        # الكود الأصلي: if maximum > 1 → normalize
        if col_max > 1 and (col_max - col_min) > 0:
            # BUG FIX: الأصل كان بيعمل zero للقيم الصفرية بس ده مش صح دائماً
            X_norm[:, i] = (col - col_min) / (col_max - col_min)
        # else: الأعمدة بين 0 و1 تفضل زي ما هي
    
    return X_norm


# اختبار
test_data = np.array([[1, 100, 0.5], [2, 200, 0.3], [3, 50, 0.8]], dtype=np.float32)
normalized = normalization(test_data)
print("اختبار Normalization:")
print(f"قبل: {test_data}")
print(f"بعد: {normalized}")
print("✅ دالة Normalization تعمل")

---
## 🏗️ Cell 6: بناء شبكة LSTM
### (ترجمة `Untitled.m`)

**البنية:**
- LSTM Layer (128 units) → Dropout
- LSTM Layer (64 units) → Dropout  
- Fully Connected (numClasses) → Dropout
- Fully Connected (numClasses) → Softmax

In [ ]:
class LSTMClassifier(nn.Module):
    """
    شبكة LSTM للتصنيف - مطابقة لبنية MATLAB.
    
    البنية:
    Input → LSTM(128, sequence) → Dropout(0.01)
           → LSTM(64, last) → Dropout(0.01)
           → FC(numClasses) → Dropout(0.01)
           → FC(numClasses) → Softmax
    """
    
    def __init__(self, input_size, num_classes, hidden1=128, hidden2=64, dropout=0.01):
        super(LSTMClassifier, self).__init__()
        
        self.input_size = input_size
        self.num_classes = num_classes
        
        # LSTM Layer 1: OutputMode='sequence' → return_sequences=True
        self.lstm1 = nn.LSTM(input_size, hidden1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        
        # LSTM Layer 2: OutputMode='last' → return_sequences=False
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        
        # Fully Connected Layer 1
        self.fc1 = nn.Linear(hidden2, num_classes)
        self.dropout3 = nn.Dropout(dropout)
        
        # Fully Connected Layer 2
        self.fc2 = nn.Linear(num_classes, num_classes)
        
        # Softmax
        self.softmax = nn.Softmax(dim=1)
    
    def forward(self, x):
        # x shape: (batch, seq_len, input_size)
        out, _ = self.lstm1(x)         # (batch, seq_len, 128)
        out = self.dropout1(out)
        
        out, _ = self.lstm2(out)        # (batch, seq_len, 64)
        out = out[:, -1, :]             # أخذ آخر timestep فقط (OutputMode='last')
        out = self.dropout2(out)
        
        out = self.fc1(out)             # (batch, num_classes)
        out = self.dropout3(out)
        
        out = self.fc2(out)             # (batch, num_classes)
        out = self.softmax(out)
        return out
    
    def get_all_weights(self):
        """جمع كل الأوزان كـ dict - مفيد للـ CBOA"""
        return {
            'lstm1_input_weights': self.lstm1.weight_ih_l0.data.clone(),
            'lstm1_recurrent_weights': self.lstm1.weight_hh_l0.data.clone(),
            'lstm1_bias': self.lstm1.bias_ih_l0.data.clone(),
            'lstm2_input_weights': self.lstm2.weight_ih_l0.data.clone(),
            'lstm2_recurrent_weights': self.lstm2.weight_hh_l0.data.clone(),
            'lstm2_bias': self.lstm2.bias_ih_l0.data.clone(),
            'fc1_weights': self.fc1.weight.data.clone(),
            'fc1_bias': self.fc1.bias.data.clone(),
            'fc2_weights': self.fc2.weight.data.clone(),
            'fc2_bias': self.fc2.bias.data.clone(),
        }
    
    def set_weights(self, weight_dict):
        """تحديث الأوزان من dict"""
        self.lstm1.weight_ih_l0.data = weight_dict['lstm1_input_weights']
        self.lstm1.weight_hh_l0.data = weight_dict['lstm1_recurrent_weights']
        self.lstm1.bias_ih_l0.data = weight_dict['lstm1_bias']
        self.lstm2.weight_ih_l0.data = weight_dict['lstm2_input_weights']
        self.lstm2.weight_hh_l0.data = weight_dict['lstm2_recurrent_weights']
        self.lstm2.bias_ih_l0.data = weight_dict['lstm2_bias']
        self.fc1.weight.data = weight_dict['fc1_weights']
        self.fc1.bias.data = weight_dict['fc1_bias']
        self.fc2.weight.data = weight_dict['fc2_weights']
        self.fc2.bias.data = weight_dict['fc2_bias']


# اختبار البنية
test_model = LSTMClassifier(input_size=122, num_classes=5)
print("🏗️ بنية الشبكة:")
print(test_model)
total_params = sum(p.numel() for p in test_model.parameters())
print(f"\n📊 إجمالي المعاملات: {total_params:,}")

---
## 📏 Cell 7: دوال تقييم الأداء
### (ترجمة `NMSE.m` إلى `NMSE9.m`)

**الشرح:** كل NMSE دالة في الأصل كانت بتحدّث نوع مختلف من الأوزان وتحسب الـ Accuracy والـ Cost.
في Python بندمج ده في دالة واحدة مرنة.

In [ ]:
def evaluate_model(model, X_test, y_test_encoded):
    """
    تقييم أداء النموذج - بديل موحد لكل دوال NMSE.
    
    Returns:
        acc (float): دقة التصنيف
        cost (float): قيمة الـ NMSE (خسارة)
    """
    model.eval()
    with torch.no_grad():
        # تحويل البيانات
        if isinstance(X_test, np.ndarray):
            X_tensor = torch.FloatTensor(X_test).unsqueeze(1)  # (N, 1, features)
        else:
            X_tensor = X_test
        
        outputs = model(X_tensor)
        preds = outputs.argmax(dim=1).numpy()
        
        # Accuracy
        acc = accuracy_score(y_test_encoded, preds)
        
        # NMSE: Mean Squared Error normalized
        cost = np.mean((preds - y_test_encoded) ** 2) / len(y_test_encoded)
    
    return acc, cost


def make_cost_function(model, weight_key, X_test, y_test_encoded):
    """
    بناء cost function لـ CBOA لنوع معين من الأوزان.
    بديل لدوال NMSE0 - NMSE9.
    
    Parameters:
        model: النموذج
        weight_key (str): اسم الوزن المراد تحسينه
        X_test: بيانات الاختبار
        y_test_encoded: التصنيفات (numeric)
    
    Returns:
        cost_fn: دالة (weights -> acc, cost)
    """
    def cost_fn(new_weights):
        # حفظ الأوزان الحالية
        original_weights = model.get_all_weights()
        
        # تحديث الوزن المحدد
        updated = copy.deepcopy(original_weights)
        updated[weight_key] = torch.FloatTensor(new_weights)
        model.set_weights(updated)
        
        # تقييم
        acc, cost = evaluate_model(model, X_test, y_test_encoded)
        
        return acc, cost
    
    return cost_fn


print("✅ دوال التقييم جاهزة")

---
## 🦋 Cell 8: خوارزمية CBOA الرئيسية
### (ترجمة `CBOAoptimization.m`)

**كيف تعمل الخوارزمية:**
1. تبدأ بمجموعة من الحلول (butterflies/particles)
2. كل فراشة تحسب رائحتها (fragrance) بناءً على جودتها
3. احتمال p: الفراشة تتحرك نحو الأفضل عالمياً (Global Search)
4. احتمال (1-p): الفراشة تتحرك بشكل عشوائي محلياً (Local Search)
5. الفوضى (Chaos) بتحدد قيمة r في كل iteration بدلاً من rand

### ⚠️ أخطاء في الكود الأصلي (تم إصلاحها):
1. **BUG**: `particle(i).Position = nVar` → كل الـ particles بتبدأ من نفس المكان! (يجب تهيئة عشوائية)
2. **BUG**: في الـ Local Search، `particle(i).Position(JK(1))` و`JK(2)` → إذا nVar كانت 1D فده بيرجع scalar مش vector
3. **BUG**: `ChaosVec` تتملى بـ 10 صفوف بس بيتقرأ كـ `ChaosVec(it)` مش `ChaosVec(row, it)`

In [ ]:
def cboa_optimization(cost_function, initial_weights,
                       max_iter=10, n_pop=30, p=0.8,
                       power_exponent=0.1, sensory_modality=0.01,
                       chaos_type=2, verbose=True):
    """
    Chaotic Butterfly Optimization Algorithm (CBOA)
    ترجمة وتصحيح CBOAoptimization.m
    
    Parameters:
        cost_function: دالة (x → acc, cost) — تقيّم جودة الحل
        initial_weights (np.array): نقطة البداية (شكل الأوزان)
        max_iter (int): عدد التكرارات الأقصى
        n_pop (int): حجم المجموعة (عدد الفراشات)
        p (float): احتمال البحث العالمي مقابل المحلي
        power_exponent (float): أس حساب الرائحة
        sensory_modality (float): معامل الحساسية
        chaos_type (int): نوع الخريطة الفوضوية (1-10)
        verbose (bool): طباعة التقدم
    
    Returns:
        best_position: أفضل أوزان وُجدت
        best_cost: أقل خسارة وُجدت
        best_costs_history: تاريخ الخسارة عبر الـ iterations
        best_acc_history: تاريخ الدقة عبر الـ iterations
    """
    weights_shape = initial_weights.shape
    var_min = initial_weights.min()
    var_max = initial_weights.max()
    
    # إنشاء متجه الفوضى
    chaos_vec = chaos(chaos_type, 1, max_iter)

    # =====================
    # تهيئة المجموعة
    # =====================
    # BUG FIX: الأصل كان position = nVar (نفس لكل الـ particles)
    # الصحيح: تهيئة عشوائية في نطاق [var_min, var_max]
    positions = [
        np.random.uniform(var_min, var_max, weights_shape)
        for _ in range(n_pop)
    ]
    
    # تقييم أولي
    accs = []
    costs = []
    for pos in positions:
        acc, cost = cost_function(pos)
        accs.append(acc)
        costs.append(cost)
    
    # أفضل حل شخصي لكل فراشة
    best_personal_pos = [p.copy() for p in positions]
    best_personal_acc = accs.copy()
    best_personal_cost = costs.copy()
    
    # أفضل حل عالمي
    best_idx = np.argmax(accs)
    global_best_pos = positions[best_idx].copy()
    global_best_acc = accs[best_idx]
    global_best_cost = costs[best_idx]
    
    best_costs_history = []
    best_acc_history = []
    
    # =====================
    # الحلقة الرئيسية
    # =====================
    for it in range(max_iter):
        r_value = chaos_vec[it]  # قيمة فوضوية بدل rand
        
        for i in range(n_pop):
            pos = positions[i].copy()
            cost_i = costs[i]
            
            # حساب الرائحة (Fragrance) - Eq. (1)
            fp = sensory_modality * (cost_i ** power_exponent)
            
            if r_value < p:
                # ===== Global Search: التحرك نحو الأفضل =====
                # Eq. (2): fi = r^2 * GlobalBest - xi * fp
                fi = (r_value**2 * global_best_pos - pos) * fp
                new_pos = pos + fi
            else:
                # ===== Local Search: البحث المحلي العشوائي =====
                # Eq. (3)
                epsilon = r_value
                # BUG FIX: الأصل كان يأخذ عناصر بـ JK(1), JK(2) من nVar
                # وده منطقياً بيقارن عنصرين من نفس الـ position vector
                flat = pos.flatten()
                n = len(flat)
                jk = np.random.permutation(n)
                j1, j2 = jk[0] if n > 0 else 0, jk[1] if n > 1 else 0
                fi = (epsilon**2 * flat[j1] - flat[j2]) * fp
                new_pos = pos + fi
            
            # تطبيق حدود المتغيرات
            new_pos = np.clip(new_pos, var_min, var_max)
            positions[i] = new_pos
            
            # التقييم
            acc_new, cost_new = cost_function(new_pos)
            accs[i] = acc_new
            costs[i] = cost_new
            
            # تحديث الأفضل الشخصي
            if acc_new > best_personal_acc[i]:
                best_personal_pos[i] = new_pos.copy()
                best_personal_acc[i] = acc_new
                best_personal_cost[i] = cost_new
                
                # تحديث الأفضل العالمي
                if acc_new > global_best_acc:
                    global_best_pos = new_pos.copy()
                    global_best_acc = acc_new
                    global_best_cost = cost_new
        
        best_costs_history.append(global_best_cost)
        best_acc_history.append(global_best_acc)
        
        if verbose:
            print(f"Iteration {it+1}/{max_iter}: Best Acc = {global_best_acc:.4f}, Cost = {global_best_cost:.6f}")
    
    return global_best_pos, global_best_cost, best_costs_history, best_acc_history


print("✅ خوارزمية CBOA جاهزة")

---
## 📈 Cell 9: مصفوفة الالتباس (Confusion Matrix)
### (ترجمة `confusion1.m`)

**الشرح:** حساب كل مقاييس الأداء: Accuracy, Precision, Recall, F1, MCC, Kappa

In [ ]:
def compute_confusion_metrics(y_true, y_pred, class_names=None, display=True):
    """
    حساب وعرض مقاييس الأداء - ترجمة confusion1.m
    
    Parameters:
        y_true: التصنيفات الحقيقية
        y_pred: التصنيفات المتوقعة
        class_names: أسماء الكلاسات (اختياري)
        display (bool): عرض الرسومات
    
    Returns:
        result (dict): نتائج الأداء الكلي
        ref_result (dict): نتائج لكل كلاس
    """
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]
    
    # حساب TP, FP, FN, TN لكل كلاس
    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = cm.sum() - (TP + FP + FN)
    
    P = TP + FN
    N = FP + TN
    
    # المقاييس لكل كلاس
    epsilon = 1e-10  # تجنب قسمة على صفر
    sensitivity = TP / (P + epsilon)   # Recall
    specificity = TN / (N + epsilon)
    precision   = TP / (TP + FP + epsilon)
    fpr = 1 - specificity
    fnr = 1 - sensitivity
    
    # F1 Score
    f1 = 2 * (precision * sensitivity) / (precision + sensitivity + epsilon)
    
    # MCC
    mcc_num = TP * TN - FP * FN
    mcc_den = np.sqrt((TP + FP) * P * N * (TN + FN) + epsilon)
    mcc = mcc_num / mcc_den
    
    # Accuracy الكلية
    overall_acc = accuracy_score(y_true, y_pred)
    
    # Kappa
    from sklearn.metrics import cohen_kappa_score
    try:
        kappa = cohen_kappa_score(y_true, y_pred)
    except:
        kappa = 0.0
    
    result = {
        'Accuracy': overall_acc,
        'Sensitivity (Recall)': sensitivity.mean(),
        'Specificity': specificity.mean(),
        'Precision': precision.mean(),
        'F1_Score': f1.mean(),
        'MCC': mcc.mean(),
        'Kappa': kappa,
        'FalsePositiveRate': fpr.mean(),
        'FalseNegativeRate': fnr.mean(),
    }
    
    ref_result = {
        'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
        'Precision': precision,
        'F1_Score': f1,
        'MCC': mcc,
    }
    
    if display:
        labels = class_names if class_names else [str(i) for i in range(n_classes)]
        
        fig, axes = plt.subplots(1, 2, figsize=(16, 5))
        
        # مصفوفة الالتباس
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
        axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
        axes[0].tick_params(axis='x', rotation=45)
        
        # مقاييس لكل كلاس
        metrics_df = pd.DataFrame({
            'Sensitivity': sensitivity,
            'Precision': precision,
            'F1-Score': f1,
            'Specificity': specificity,
        }, index=labels)
        metrics_df.plot(kind='bar', ax=axes[1], colormap='tab10', alpha=0.8)
        axes[1].set_title('Per-Class Metrics', fontsize=13, fontweight='bold')
        axes[1].set_xlabel('Class')
        axes[1].set_ylabel('Score')
        axes[1].legend(loc='lower right')
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].set_ylim(0, 1.1)
        
        plt.tight_layout()
        plt.show()
        
        print("\n📊 النتائج الكلية:")
        for k, v in result.items():
            print(f"  {k}: {v:.4f}")
    
    return result, ref_result


print("✅ دوال الـ Confusion Matrix جاهزة")

---
## 🚀 Cell 10: تحميل البيانات وتدريب النموذج
### (ترجمة `optimizationrandom.m` و `Untitled.m`)

⚠️ **ملاحظة:** تحتاج ملفات `KDDTrain+.txt` و `KDDTest+.txt` من NSL-KDD Dataset.
حملّهم من: https://www.unb.ca/cic/datasets/nsl.html

In [ ]:
# ================================================================
# ⚙️ الإعدادات - غيّر المسارات حسب موقع الملفات عندك
# ================================================================
TRAIN_PATH = 'KDDTrain+.txt'  # ← غيّر المسار
TEST_PATH  = 'KDDTest+.txt'   # ← غيّر المسار
MODE = 'multi'  # 'binary' أو 'multi'

# ================================================================
# تحميل البيانات
# ================================================================
print("📥 تحميل البيانات...")
try:
    X_train_raw, y_train_raw = preprocess_kdd(TRAIN_PATH, mode=MODE)
    X_test_raw,  y_test_raw  = preprocess_kdd(TEST_PATH,  mode=MODE)
    print(f"✅ Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")
    data_loaded = True
except FileNotFoundError:
    print("⚠️ ملفات KDD غير موجودة - سيتم استخدام بيانات وهمية للتوضيح")
    # بيانات وهمية للتوضيح
    N_TRAIN, N_TEST, N_FEAT = 1000, 200, 122
    N_CLASSES = 5 if MODE == 'multi' else 2
    X_train_raw = np.random.rand(N_TRAIN, N_FEAT).astype(np.float32)
    y_train_raw = np.random.choice(['normal','DOS','Probe','R2L','U2R'][:N_CLASSES], N_TRAIN)
    X_test_raw  = np.random.rand(N_TEST,  N_FEAT).astype(np.float32)
    y_test_raw  = np.random.choice(['normal','DOS','Probe','R2L','U2R'][:N_CLASSES], N_TEST)
    data_loaded = False

# ================================================================
# مزامنة الأعمدة بين Train و Test (مهم بعد One-Hot Encoding)
# ================================================================
# PyTorch بيحتاج نفس عدد الـ features في train و test
n_feat_train = X_train_raw.shape[1]
n_feat_test  = X_test_raw.shape[1]
n_feat = max(n_feat_train, n_feat_test)

if n_feat_train < n_feat:
    X_train_raw = np.pad(X_train_raw, ((0,0),(0,n_feat-n_feat_train)))
if n_feat_test < n_feat:
    X_test_raw = np.pad(X_test_raw, ((0,0),(0,n_feat-n_feat_test)))

# ================================================================
# Normalization
# ================================================================
print("📊 تطبيع البيانات...")
X_train = normalization(X_train_raw)
X_test  = normalization(X_test_raw)

# ================================================================
# تحويل التصنيفات
# ================================================================
le = LabelEncoder()
le.fit(np.concatenate([y_train_raw, y_test_raw]))
y_train_enc = le.transform(y_train_raw)
y_test_enc  = le.transform(y_test_raw)
class_names = le.classes_
num_classes = len(class_names)

print(f"\n📌 ملخص البيانات:")
print(f"   Features: {n_feat}")
print(f"   Classes: {num_classes} → {list(class_names)}")
print(f"   Train size: {X_train.shape[0]}")
print(f"   Test size:  {X_test.shape[0]}")

# توزيع الكلاسات
unique, counts = np.unique(y_train_raw, return_counts=True)
plt.figure(figsize=(8, 4))
plt.bar(unique, counts, color=plt.cm.tab10(np.linspace(0, 1, len(unique))))
plt.title('توزيع الكلاسات في بيانات التدريب', fontsize=12)
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 🎓 Cell 11: التدريب الأولي للشبكة (Adam Optimizer)

In [ ]:
# ================================================================
# إنشاء DataLoaders
# ================================================================
MINI_BATCH_SIZE = 265  # نفس القيمة في الكود الأصلي
MAX_EPOCHS = 20        # قليل للتجربة السريعة (الأصل 100)

# تحويل لـ PyTorch tensors
# LSTM يحتاج (batch, seq_len, features) → نضيف seq_len=1
X_train_t = torch.FloatTensor(X_train).unsqueeze(1)
y_train_t = torch.LongTensor(y_train_enc)
X_test_t  = torch.FloatTensor(X_test).unsqueeze(1)
y_test_t  = torch.LongTensor(y_test_enc)

train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset  = TensorDataset(X_test_t,  y_test_t)

train_loader = DataLoader(train_dataset, batch_size=MINI_BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=MINI_BATCH_SIZE, shuffle=False)

# ================================================================
# إنشاء النموذج والمحسّن
# ================================================================
model = LSTMClassifier(input_size=n_feat, num_classes=num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # نفس الإعدادات الأصلية
criterion = nn.CrossEntropyLoss()

# ================================================================
# حلقة التدريب
# ================================================================
print(f"🎓 بدء التدريب الأولي ({MAX_EPOCHS} epochs)...")
train_losses = []
val_accs = []

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0
    
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        
        # Gradient Clipping (GradientThreshold=1 في الكود الأصلي)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    # تقييم على بيانات الاختبار
    model.eval()
    all_preds = []
    with torch.no_grad():
        for Xb, yb in test_loader:
            Xb = Xb.to(device)
            out = model(Xb)
            all_preds.extend(out.argmax(1).cpu().numpy())
    
    val_acc = accuracy_score(y_test_enc, all_preds)
    avg_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_loss)
    val_accs.append(val_acc)
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{MAX_EPOCHS} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

# رسم منحنى التدريب
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_title('Training Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(val_accs, 'g-', linewidth=2)
axes[1].set_title('Validation Accuracy', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].grid(True, alpha=0.3)

plt.suptitle('📈 Training Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ دقة النموذج الأولي: {val_accs[-1]:.4f}")

---
## 🦋 Cell 12: تحسين الأوزان بـ CBOA
### (ترجمة `optimizationrandom.m` - الجزء الرئيسي)

**الترتيب:** نحسّن كل نوع من الأوزان بشكل متسلسل.

In [ ]:
# ================================================================
# تهيئة الأوزان العشوائية (نفس optimizationrandom.m)
# ================================================================
print("🎲 تهيئة الأوزان عشوائياً في نطاق [-5, 5]...")

# نسخة محسّنة من الشبكة بأوزان عشوائية
model_optimized = copy.deepcopy(model)

for param in model_optimized.parameters():
    nn.init.uniform_(param.data, -5, 5)  # نفس unifrnd(-5, 5, ...) في الأصل

# تقييم قبل CBOA
acc_before, _ = evaluate_model(model_optimized, X_test, y_test_enc)
print(f"📊 دقة قبل CBOA (أوزان عشوائية): {acc_before:.4f}")

# ================================================================
# تطبيق CBOA على كل نوع من الأوزان
# ================================================================
WEIGHT_KEYS = [
    ('lstm1_input_weights',     'LSTM1 - InputWeights'),
    ('lstm1_recurrent_weights', 'LSTM1 - RecurrentWeights'),
    ('lstm1_bias',              'LSTM1 - Bias'),
    ('lstm2_input_weights',     'LSTM2 - InputWeights'),
    ('lstm2_recurrent_weights', 'LSTM2 - RecurrentWeights'),
    ('lstm2_bias',              'LSTM2 - Bias'),
    ('fc1_weights',             'FC1 - Weights'),
    ('fc1_bias',                'FC1 - Bias'),
    ('fc2_weights',             'FC2 - Weights'),
    ('fc2_bias',                'FC2 - Bias'),
]

all_cost_histories = {}
start_time = time.time()

for key, name in WEIGHT_KEYS:
    print(f"\n{'='*50}")
    print(f"🔧 تحسين: {name}")
    print(f"{'='*50}")
    
    # الأوزان الحالية كنقطة بداية
    current_weights = model_optimized.get_all_weights()
    initial_w = current_weights[key].numpy()
    
    # بناء cost function
    cost_fn = make_cost_function(model_optimized, key, X_test, y_test_enc)
    
    # تشغيل CBOA
    best_w, best_cost, cost_hist, acc_hist = cboa_optimization(
        cost_function=cost_fn,
        initial_weights=initial_w,
        max_iter=10,   # نفس MaxIt في الأصل
        n_pop=10,      # قليل للسرعة (الأصل 30)
        p=0.8,
        chaos_type=2,  # Circle map
        verbose=True
    )
    
    # تطبيق أفضل الأوزان
    updated = model_optimized.get_all_weights()
    updated[key] = torch.FloatTensor(best_w)
    model_optimized.set_weights(updated)
    
    all_cost_histories[name] = cost_hist

elapsed = time.time() - start_time
print(f"\n⏱️ وقت التنفيذ الكلي: {elapsed:.1f} ثانية")

# تقييم بعد CBOA
acc_after, _ = evaluate_model(model_optimized, X_test, y_test_enc)
print(f"\n📊 دقة بعد CBOA: {acc_after:.4f}")
print(f"📈 تحسن في الدقة: {(acc_after - acc_before)*100:.2f}%")

---
## 📊 Cell 13: النتائج والتقييم النهائي

In [ ]:
# ================================================================
# التنبؤ النهائي
# ================================================================
model_optimized.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).unsqueeze(1)
    outputs = model_optimized(X_test_tensor)
    y_pred = outputs.argmax(dim=1).numpy()

final_acc = accuracy_score(y_test_enc, y_pred)
print(f"🎯 الدقة النهائية: {final_acc:.4f} ({final_acc*100:.2f}%)")

# ================================================================
# Confusion Matrix ومقاييس الأداء
# ================================================================
result, ref_result = compute_confusion_metrics(
    y_test_enc, y_pred, 
    class_names=class_names,
    display=True
)

# ================================================================
# رسم منحنى التقارب لكل نوع من الأوزان
# ================================================================
n_plots = len(all_cost_histories)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, (name, history) in enumerate(all_cost_histories.items()):
    axes[idx].plot(history, color=plt.cm.tab10(idx/10), linewidth=2)
    axes[idx].set_title(name, fontsize=8)
    axes[idx].set_xlabel('Iteration')
    axes[idx].set_ylabel('Cost')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('🦋 منحنى تقارب CBOA لكل نوع من الأوزان', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ================================================================
# مقارنة قبل وبعد التحسين
# ================================================================
# دقة النموذج الأصلي
with torch.no_grad():
    out_orig = model(X_test_tensor)
    y_pred_orig = out_orig.argmax(1).numpy()
acc_orig = accuracy_score(y_test_enc, y_pred_orig)

print("\n" + "="*50)
print("📊 ملخص المقارنة النهائية")
print("="*50)
models_comparison = {
    'Adam (Original)': acc_orig,
    'CBOA Optimized': final_acc,
}

plt.figure(figsize=(8, 5))
bars = plt.bar(models_comparison.keys(), 
               [v*100 for v in models_comparison.values()],
               color=['#4CAF50', '#2196F3'], alpha=0.8, width=0.4)
plt.ylabel('Accuracy (%)')
plt.title('مقارنة الدقة: Adam vs CBOA', fontsize=13, fontweight='bold')
plt.ylim(0, 110)
for bar, (name, val) in zip(bars, models_comparison.items()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val*100:.2f}%', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🔁 Cell 14: إعادة التدريب بالأوزان المحسّنة
### (ترجمة الجزء الثاني من `optimizationrandom.m`)

In [ ]:
# ================================================================
# إعادة التدريب بالأوزان المحسّنة بـ CBOA
# ================================================================
print("🔁 إعادة التدريب بالأوزان المحسّنة...")

# نسخة من النموذج المحسّن
model_retrained = copy.deepcopy(model_optimized)
optimizer2 = torch.optim.Adam(model_retrained.parameters(), lr=0.001)

MAX_EPOCHS2 = 10  # أقل لأن الأوزان مبدئية جيدة
train_losses2 = []
val_accs2 = []

for epoch in range(MAX_EPOCHS2):
    model_retrained.train()
    epoch_loss = 0
    
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer2.zero_grad()
        outputs = model_retrained(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_retrained.parameters(), 1.0)
        optimizer2.step()
        epoch_loss += loss.item()
    
    model_retrained.eval()
    preds_all = []
    with torch.no_grad():
        for Xb, _ in test_loader:
            preds_all.extend(model_retrained(Xb.to(device)).argmax(1).cpu().numpy())
    
    val_acc2 = accuracy_score(y_test_enc, preds_all)
    train_losses2.append(epoch_loss / len(train_loader))
    val_accs2.append(val_acc2)
    print(f"Epoch {epoch+1}/{MAX_EPOCHS2} | Loss: {train_losses2[-1]:.4f} | Val Acc: {val_acc2:.4f}")

# النتائج النهائية
final_acc2 = val_accs2[-1]
print(f"\n🏆 دقة النموذج النهائي (CBOA + Adam): {final_acc2:.4f} ({final_acc2*100:.2f}%)")

# مقارنة شاملة
print("\n" + "="*60)
print("🏆 المقارنة الشاملة للنتائج")
print("="*60)
print(f"{'Model':<35} {'Accuracy':>10}")
print("-"*45)
print(f"{'LSTM (Adam only)':<35} {acc_orig*100:>9.2f}%")
print(f"{'LSTM (CBOA weights only)':<35} {final_acc*100:>9.2f}%")
print(f"{'LSTM (CBOA + Adam fine-tune)':<35} {final_acc2*100:>9.2f}%")
print("="*60)

---
## 🐛 Cell 15: ملخص الأخطاء في الكود الأصلي

In [ ]:
bugs_summary = """
╔══════════════════════════════════════════════════════════════════╗
║          🐛 ملخص الأخطاء في الكود الأصلي (MATLAB)              ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║ 1. [CBOAoptimization.m] - BUG CRITICAL                          ║
║    particle(i).Position = nVar                                   ║
║    ❌ كل الـ particles بتبدأ من نفس الموقع (nVar نفسها)!        ║
║    ✅ الصح: تهيئة عشوائية unifrnd(VarMin, VarMax, VarSize)      ║
║                                                                  ║
║ 2. [CBOAoptimization.m] - BUG MEDIUM                            ║
║    ChaosVec = zeros(1, MaxIt)                                    ║
║    ثم: for i=1:10 ChaosVec(i,:) = chaos(...)                    ║
║    ❌ بيكتب في صف واحد فقط (1, MaxIt) مش matrix كاملة          ║
║    ✅ المقصود: ChaosVec = zeros(10, MaxIt)                       ║
║                                                                  ║
║ 3. [Nominal2Numbersbinary.m] - BUG NAMING                       ║
║    اسم الملف: Nominal2Numbersbinary.m                           ║
║    اسم الدالة داخل الملف: Nominal2Numbers                        ║
║    ❌ MATLAB بيقرأ اسم الدالة من الملف مش من التعريف الداخلي   ║
║    ✅ لازم يكونوا متطابقين                                       ║
║                                                                  ║
║ 4. [Normalization.m] - BUG MINOR                                ║
║    matrix_normalized(j,i) = (selected_column(j)-minimum)/...    ║
║    ❌ لو maximum == minimum → قسمة على صفر (NaN)               ║
║    ✅ لازم تتحقق أن (maximum - minimum) > 0                     ║
║                                                                  ║
║ 5. [confusion1.m] - BUG VALIDATION                              ║
║    بيشترط أن un_actual == un_predict                             ║
║    ❌ لو النموذج ما تنبأش بكلاس معين → Error                   ║
║    ✅ لازم تعمل union للكلاسات مش intersection                  ║
║                                                                  ║
║ 6. [CBOAoptimization.m] - BUG LOCAL SEARCH                      ║
║    particle(i).Position(JK(1)) و JK(2)                          ║
║    ❌ لو الـ weights 2D matrix، الـ indexing ده غلط             ║
║    ✅ لازم تعمل flatten أول                                      ║
║                                                                  ║
║ 7. [Untitled.m + optimizationrandom.m] - CODE SMELL             ║
║    classificationLayer() بدون Classes parameter                  ║
║    ❌ ممكن يتعب في بعض إصدارات MATLAB                          ║
║    ✅ أفضل: classificationLayer('Classes', C)                   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(bugs_summary)

---
## 💾 Cell 16: حفظ النموذج

In [ ]:
# حفظ النموذج المحسّن
torch.save({
    'model_state_dict': model_retrained.state_dict(),
    'class_names': class_names,
    'n_features': n_feat,
    'num_classes': num_classes,
    'final_accuracy': final_acc2,
}, 'cboa_lstm_model.pth')

print("✅ النموذج محفوظ في: cboa_lstm_model.pth")
print(f"📊 الدقة النهائية المحفوظة: {final_acc2:.4f}")

# لتحميل النموذج لاحقاً:
# checkpoint = torch.load('cboa_lstm_model.pth')
# model_loaded = LSTMClassifier(checkpoint['n_features'], checkpoint['num_classes'])
# model_loaded.load_state_dict(checkpoint['model_state_dict'])